# 본체(body) 재학습 — cube_v7, 6클래스 태극기 깃발 포함

4종 도형 + `fruit_photo_cube` + `arrival` YOLOv8n. 기존 `models/cube.pt`를 대체할 본체 카메라용 모델입니다.

- 업로드: `dataset_body_flag_v7.zip` (내부 루트 `dataset/{images,labels}`)
- 클래스 순서: `cube`, `octahedron`, `dodecahedron`, `icosahedron`, `fruit_photo_cube`, `arrival`
- `yolov8n.pt`에서 새로 학습합니다. 기존 `cube.pt` 이어학습은 사용하지 않습니다.
- 결과 `best.pt`를 젯슨의 `models/cube.pt`로 배포합니다.

In [ ]:
# [셀1] 업로드 + 압축해제
from google.colab import files
import os, shutil
up = files.upload()
ZIP = next(iter(up))
shutil.rmtree('/content/dataset', ignore_errors=True)
!unzip -o -q "$ZIP" -d /content
print('uploaded:', ZIP, '-> extracted:', sorted(os.listdir('/content/dataset')))

In [ ]:
# [셀2] train/val 분리 + data_colab.yaml 생성
import os, glob, random, shutil, yaml
random.seed(0)
ROOT = '/content/dataset'
CLASSES = ['cube', 'octahedron', 'dodecahedron', 'icosahedron', 'fruit_photo_cube', 'arrival']
HELD_OUT_VAL = True
VAL_FRAC = 0.15
lbl = lambda p: '/content/dataset/labels/' + os.path.splitext(os.path.basename(p))[0] + '.txt'
pairs = [(i, lbl(i)) for i in sorted(glob.glob('/content/dataset/images/*')) if os.path.exists(lbl(i))]
print('pairs:', len(pairs), '(빈 라벨=배경음성 포함)')
if HELD_OUT_VAL:
    random.shuffle(pairs); n = int(len(pairs) * VAL_FRAC)
    for split, items in [('train', pairs[n:]), ('val', pairs[:n])]:
        for s in ('images', 'labels'):
            os.makedirs(f'/content/dataset/{split}/{s}', exist_ok=True)
        for img, lb in items:
            shutil.copy(img, f'/content/dataset/{split}/images/')
            shutil.copy(lb, f'/content/dataset/{split}/labels/')
    print('train', len(pairs) - n, '/ val', n)
    data = dict(path=ROOT, train='train/images', val='val/images')
else:
    data = dict(path=ROOT, train='images', val='images')
data.update(nc=len(CLASSES), names=CLASSES)
yaml.safe_dump(data, open('/content/dataset/data_colab.yaml', 'w'))
print(open('/content/dataset/data_colab.yaml').read())

In [ ]:
# [셀3] 학습 — 본체용 imgsz=640
!pip -q install ultralytics
from ultralytics import YOLO
YOLO('yolov8n.pt').train(data='/content/dataset/data_colab.yaml',
    epochs=100, imgsz=640, batch=16, patience=30, name='cube_v7_flag')

In [ ]:
# [셀4] best.pt 내려받기
from google.colab import files
files.download('runs/detect/cube_v7_flag/weights/best.pt')